# 04 — Entrenamiento clasificador TOPS (Local)

Versión local del notebook de training de tops. No requiere Colab ni GCS.

**Pre-requisito:** ejecutar `02_prepare_data_local.ipynb` primero.

**Hardware:** Optimizado para RTX 3050 (4GB VRAM) con batch_size=8.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('.').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.colab import setup_local
config = setup_local(repo_root=REPO_ROOT)

config['tops']['training']['epochs'] = 15
config['tops']['training']['batch_size'] = 8
config['tops']['training']['num_workers'] = 4
config['tops']['training']['use_class_weights'] = True

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

print(f'\nBackbone: {config["tops"]["backbone"]}')
print(f'Batch size: {config["tops"]["training"]["batch_size"]}')
print(f'Heads: {list(config["tops"]["heads"].keys())}')

## 2. Verificar datos

In [ ]:
import os

train_csv = f"{config['paths']['splits']}/tops_1_train.csv"
val_csv   = f"{config['paths']['splits']}/tops_1_val.csv"
cache_dir = config['paths']['images_tops']

print('=== Verificación de datos ===')
for label, path in [('Train CSV', train_csv), ('Val CSV', val_csv)]:
    exists = os.path.exists(path)
    print(f'  {label}: {"✓" if exists else "✗ FALTA"} — {path}')

n_imgs = len(list(Path(cache_dir).glob('*.jpg')))
print(f'  Imágenes en cache: {n_imgs:,} — {cache_dir}')

if not os.path.exists(train_csv):
    raise FileNotFoundError(f'No se encontró {train_csv}. Ejecutá 02_prepare_data_local.ipynb primero.')

## 3. Entrenar

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

from src.classification.train import train_from_csv

result = train_from_csv(
    config=config, dataset_key='tops',
    train_csv=train_csv, val_csv=val_csv, cache_dir=cache_dir,
    history_path=f"{config['paths']['outputs']}/history_tops.json",
)
print(f"\n✓ Best F1-macro: {result['best_metric']:.3f}")
print(f"✓ Checkpoint: {result['checkpoint']}")

## 4. Visualizar historia de entrenamiento

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = pd.DataFrame(result['history'])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history['epoch'], history['train_loss'], 'o-', label='train loss')
axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss'); axes[0].set_title('Train loss'); axes[0].grid(True)
axes[1].plot(history['epoch'], history['val_acc_mean'], 'o-', label='val acc')
axes[1].plot(history['epoch'], history['val_f1_mean'], 'o-', label='val F1-macro')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Metrica')
axes[1].set_title('Validation'); axes[1].legend(); axes[1].grid(True)
plt.tight_layout(); plt.show()

## 5. Métricas finales por head

In [ ]:
import pandas as pd
last = result['history'][-1]['per_head']
rows = [{'head': name, 'accuracy': round(m['accuracy'], 3), 'f1_macro': round(m['f1_macro'], 3), 'n_val': m['n']} for name, m in last.items()]
display(pd.DataFrame(rows))